<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson03/hints_3_gradio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Hints 3: A web app for your model

Hints for Assignment 02, in three parts:

1. Build your own image dataset
2. Fine-tune a model on it
3. **Turn it into a web app** ← *you are here*

[Gradio](https://www.gradio.app/) turns a Python function into a web app with a few lines of code, and [HuggingFace Spaces](https://huggingface.co/spaces) hosts such apps for free. We will go through both in the lesson.

A CPU runtime is fine for this notebook: *Runtime → Change runtime type → CPU*.

In [ ]:
from fastai.vision.all import *
import gradio as gr

---
## 1. Load your model

Upload `model.pkl` from part 2. (fastai warns that loading such a file is insecure. That is about files from strangers, this one is your own.)

In [ ]:
from google.colab import files

files.upload()                     # choose your model.pkl
learn = load_learner("model.pkl")
categories = learn.dls.vocab
categories

---
## 2. Try it on one image

Upload an image, e.g. one from your test set:

In [ ]:
uploaded = files.upload()
image_file = list(uploaded.keys())[0]
PILImage.create(image_file).show(figsize=(3, 3))
plt.show()

prediction, index, probabilities = learn.predict(PILImage.create(image_file))
for category, p in zip(categories, probabilities):
    print(f"{category:20s} {p:.1%}")

---
## 3. A Gradio app

A Gradio app has three parts:
- a Python **function**: here it gets an image and returns a dictionary *category → probability*
- its **input**: `gr.Image` lets the user upload an image
- its **output**: `gr.Label` shows the dictionary as bars

In [ ]:
def classify_image(image):
    prediction, index, probabilities = learn.predict(image)
    return dict(zip(categories, map(float, probabilities)))

classify_image(PILImage.create(image_file))     # look inside: what the function returns

The app runs right here in the notebook, and `share=True` also gives you a temporary public link. Try it on your phone!

In [ ]:
app = gr.Interface(fn=classify_image,
                   inputs=gr.Image(width=224, height=224),
                   outputs=gr.Label(),
                   examples=[image_file])
app.launch(share=True)

---
## 4. Deploy it to HuggingFace Spaces

The link above stops working when you close the notebook. A **Space** runs the app permanently:

1. Create a free [HuggingFace account](https://huggingface.co/join) and log in.
2. Create a [new Space](https://huggingface.co/new-space): choose **Gradio** as the *Space SDK*, the *Blank* template, the free *CPU basic* hardware, and make it **Public**.
3. In the **Files** tab of your Space, open `README.md` and check that it contains the line `python_version: '3.13'` (the Python version of Colab, see below). If it doesn't, edit the file and add the line, e.g. below `sdk_version`.
4. Click *Contribute → Upload files* and upload:
   - your `model.pkl`
   - `app.py` and `requirements.txt` from [lesson03/gradio_app](https://github.com/simecek/dspracticum2026/tree/main/lesson03/gradio_app) (no changes needed)
   - 2-3 `.jpg` images, e.g. from your test set (the app shows them as examples)
5. The Space starts building (*Building*, then *Running*), which takes a few minutes. Then open the **App** tab.

If you see *Build error* or *Runtime error*, click **Logs** to see what went wrong.

`app.py` is the same code as in section 3. `requirements.txt` lists the libraries the Space installs.

The Space must use the same versions of Python and of the libraries as Colab, where you trained the model. Otherwise `model.pkl` may fail to load. These are the versions in Colab:

In [ ]:
import platform
import fastai, timm, torch, torchvision

print("python_version:", platform.python_version())    # README.md (only the first two numbers, e.g. '3.13')
print()
print("fastai==" + fastai.__version__)                  # requirements.txt
print("timm==" + timm.__version__)
print("torch==" + torch.__version__.split("+")[0])
print("torchvision==" + torchvision.__version__.split("+")[0])

If they differ from your `README.md` or `requirements.txt`, change the numbers there.